# RAG Practice Exam

In [1]:
%load_ext dotenv
%dotenv

In [34]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters.character import CharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

In [3]:
ytt_api = YouTubeTranscriptApi()

In [4]:
video_id = '6Fa91SY9Gnw'

In [5]:
fetched_transcript = ytt_api.fetch(video_id)

In [6]:
transcript_striped = ''
for snippet in fetched_transcript:
    snippet.text.replace(u'\xa0',u'').replace(u'\uf0a7',u'')
    transcript_striped += snippet.text

In [7]:
print(transcript_striped)

According to the TIOBE Index for January 2022,  Python is the programming language with the 
highest increase in ratings in one year,  making it officially the top choice 
of employers and professionals alike.  In fact, it has become a key prerequisite 
to landing a job in data science. Luckily,  Python is also very beginner friendly. So, if 
you’re an aspiring data analyst or data scientist,  learning Python is certainly the way to go.
That said, our job is to make every step of your  learning curve easy and enjoyable. So, If you’re 
still new to Python, you might want to start  for free with our Introduction to Python course. 
We’ve provided a link in the description. But if  you already have some basic Python experience and 
want to take it to the next level, in this video  you’ll discover 4 interesting Python projects 
for beginners – all including code for practice! Great! So, let’s start with project 
number 1 - Uber Trips Analysis Since it was founded in 2009, Uber has become on

In [8]:
len(transcript_striped)

5964

In [9]:
spliter = CharacterTextSplitter(separator='.', chunk_size=500, chunk_overlap=0)

In [10]:
transcript_split = spliter.split_text(transcript_striped)

Created a chunk of size 590, which is longer than the specified 500


In [11]:
print(transcript_split)

['According to the TIOBE Index for January 2022,\xa0\xa0Python is the programming language with the\xa0\nhighest increase in ratings in one year,\xa0\xa0making it officially the top choice\xa0\nof employers and professionals alike.\xa0\xa0In fact, it has become a key prerequisite\xa0\nto landing a job in data science. Luckily,\xa0\xa0Python is also very beginner friendly. So, if\xa0\nyou’re an aspiring data analyst or data scientist,\xa0\xa0learning Python is certainly the way to go', 'That said, our job is to make every step of your\xa0\xa0learning curve easy and enjoyable. So, If you’re\xa0\nstill new to Python, you might want to start\xa0\xa0for free with our Introduction to Python course.\xa0\nWe’ve provided a link in the description', 'But if\xa0\xa0you already have some basic Python experience and\xa0\nwant to take it to the next level, in this video\xa0\xa0you’ll discover 4 interesting Python projects\xa0\nfor beginners – all including code for practice!\xa0Great! So, let’s star

In [12]:
len(transcript_split)

15

In [13]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    # With the `text-embedding-3` class
    # of models, you can specify the size
    # of the embeddings you want returned.
    # dimensions=1024
)



In [14]:
vectorstore = InMemoryVectorStore.from_texts(
    transcript_split,
    embedding=embeddings,
)

In [18]:
retriever = vectorstore.as_retriever(search_type = 'mmr', 
                                     search_kwargs = {'k':3, 
                                                      'lambda_mult':0.3})

In [21]:
retrieved_documents = retriever.invoke("Could you tell me all essential beginer Python project according to the video?")

In [27]:
for d in retrieved_documents:
    print(f"Doc:\n{d.page_content}\n\n")

Doc:
The exciting thing about this beginner  Python project is that you can turn it 
into a live product that helps people! This Python project will teach you how to:
 Read and explore a textual dataset  Build a machine learning model  with TfidfVectorizer
 Create a confusion matrix  and understand its components
 Evaluate the model’s accuracy The links we’ve provided in the description of 
the video would allow you to access these projects  and give them a try


Doc:
This 
unreasonable behavior caused many deaths,  especially in hospitals where most doctors—unlike 
today—never washed their hands before surgeries.  In the mid-1800s, the Hungarian physician, Dr. 
Ignaz Semmelweis, decided to do something about  this tragedy. He started investigating the reasons 
behind the deaths in Vienna general hospital  and those of women giving birth in particular


Doc:
According to the TIOBE Index for January 2022,  Python is the programming language with the 
highest increase in ratings in o

**To verify: these documents (above) reference 2 different projects**

In [31]:
message_template1 = SystemMessagePromptTemplate.from_template('''You are a helpful chatbot that answers questions on Youtube videos.
                                                    Answer the questions using only the following context:
                                                    {context}
                                                    ''')

message_template2 = HumanMessagePromptTemplate.from_template('{question}')

In [33]:
chat_template = ChatPromptTemplate([message_template1,message_template2])

In [35]:
chat = ChatOpenAI(model='gpt-6-luna', seed=365, temperature=1)